In [49]:
from openai import OpenAI
import gradio as gr
import requests 

In [50]:
response = requests.get("http://localhost:11434/")
print(response.content)

b'Ollama is running'


In [51]:
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(base_url=ollama_url, api_key="")  # No API key needed for Ollama

In [52]:
def chat(message, history):
    return "hello"

In [53]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


In [54]:
system_message = "You are a helpful assistant."

In [55]:
MODEL = "llama3.2"

def chat(message, history):

    print("Message:", message)

    history = [{"role":h["role"], "content":h["content"]} for h in history]

    print("History:", history)

    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    print(" whole Messages:", messages)

    response = ollama.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [36]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


Message: hi
History: []
 whole Messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'hi'}]
Message: my name is prathmesh
History: [{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'How can I assist you today?'}]
 whole Messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'How can I assist you today?'}, {'role': 'user', 'content': 'my name is prathmesh'}]
Message: whats my name
History: [{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'How can I assist you today?'}, {'role': 'user', 'content': 'my name is prathmesh'}, {'role': 'assistant', 'content': "Nice to meet you, Prathmesh! Is there something on your mind that you'd like to talk about or ask for help with? I'm all ears!"}]
 whole Messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'hi'}, {'role': 'assistant'

In [56]:
system_message = """
You are an expert AI assistant.

Your responsibilities:
- Explain concepts clearly.
- Give practical examples.
- Answer in a friendly tone.
- If the question is about DevOps, explain step by step.
- If the user asks about Python, include code examples.
- Never make up information.
- If you are unsure, say so.

Formatting Rules:
- Use headings.
- Use bullet points.
- Use code blocks when appropriate.
- Keep answers concise unless the user asks for detail.

...
...
...
"""

In [57]:

available_models = ["llama3.2", "gpt-oss:20b"]

def chat(message, history, model):

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": ""})

    response = ollama.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    for chunk in response:
        if chunk.choices[0].delta.content:
            history[-1]["content"] += chunk.choices[0].delta.content
            yield "", history



In [60]:
with gr.Blocks() as demo:

    model_choice = gr.Dropdown(
        choices=available_models,
        value="llama3.2",
        label="Model"
    )

    chatbot = gr.Chatbot(type="messages", height=400)

    user_input = gr.Textbox(
        placeholder="Ask something..."
    )

    clear_btn = gr.Button("Clear")

    user_input.submit(
        fn=chat,
        inputs=[user_input, chatbot, model_choice],
        outputs=[user_input, chatbot]
    )

    clear_btn.click(
        lambda: ([], ""),
        outputs=[chatbot, user_input]
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7880
* Running on public URL: https://85153cb9ab945c26b7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
